# Tuples, Dictionaries & Charts — Lock It, Look It Up, Plot It

**Software Programming (IGS1931) · Week 12**

The lecture introduced **tuples** (ordered records that can't be changed), **dictionaries** (look-up tables of `key: value` pairs) and **matplotlib** (Python's chart library). Today we start with a small puzzle that uses all three, then for each topic we ask **"why should I care?"**, review what the lecture taught with short demos, and practise by **predicting**, **debugging** and **writing** code with realistic data. At the end, your **project team** races through the **Chart Challenge**: turn real data into charts that a checker approves.

## Today You Will Be Able To...

- Explain **why** tuples, dictionaries and charts exist, with a real example for each
- Make tuples (and avoid the `(20)` trap), index them, unpack them, and know they **can't be changed**
- Store and look up data in a **dictionary**: add, change, delete, `in`, `get`, `keys`, `values`, `items`
- Count and total things with a dictionary, which is the most common dict job in real programs
- Draw **line, bar, pie, scatter** and **histogram** charts with matplotlib, with a title, labels and a legend

---

## 0. Warm-up — Is There a Shared Birthday in This Room?

There are about **40 students** in this class. What is the chance that **at least two** of us have the same birthday?

Write your guess down **before** running the cell.

Don't worry about how the code works yet: it uses a **tuple** `(month, day)` for each birthday, a **dictionary** to spot a repeat instantly, and **matplotlib** to draw the answer. By the end of today you'll understand every line.

In [ ]:
import random
import matplotlib.pyplot as plt

CLASS_SIZE = 40                                                    # ← how many of us are here today?
DAYS_IN_MONTH = (31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31)   # a TUPLE: these never change


def random_birthday():
    """Return a random birthday as a tuple (month, day)."""
    day = random.randint(1, 365)
    month = 1
    for days in DAYS_IN_MONTH:
        if day <= days:
            return (month, day)
        day = day - days
        month = month + 1


def shared_birthday(class_size):
    """Make a random class. True if two people share a birthday."""
    seen = {}                          # a DICTIONARY: {(month, day): True}
    for person in range(class_size):
        bday = random_birthday()
        if bday in seen:               # instant look-up, no searching
            return True
        seen[bday] = True
    return False


def exact_chance(n):
    """The math: 1 - (365/365 * 364/365 * ... ) for n people."""
    no_match = 1
    for i in range(n):
        no_match = no_match * (365 - i) / 365
    return 1 - no_match


# Simulate 2,000 random classes of every size from 2 to 60 people
sizes = list(range(2, 61))
chances = []
for n in sizes:
    hits = 0
    for trial in range(2000):
        if shared_birthday(n):
            hits = hits + 1
    chances.append(hits / 2000)

print('23 people:', round(exact_chance(23) * 100, 1), '%')
print(CLASS_SIZE, 'people:', round(exact_chance(CLASS_SIZE) * 100, 1), '%   (simulated:',
      round(chances[CLASS_SIZE - 2] * 100, 1), '%)')

plt.plot(sizes, chances, label='simulated (2,000 classes each)')
plt.plot(sizes, [exact_chance(n) for n in sizes], linestyle='dashed', label='exact math')
plt.axvline(CLASS_SIZE, color='red')
plt.text(CLASS_SIZE + 1, 0.3, 'our class', color='red')
plt.xlabel('people in the room')
plt.ylabel('chance of a shared birthday')
plt.title('The Birthday Paradox')
plt.legend()
plt.show()

**Let's test it for real.** Call out your birthday as `(month, day)`, with no names, and we type it in. The dictionary counts each birthday, so any count above 1 is a match.

In [ ]:
our_birthdays = [(3, 14), (11, 19), (7, 2), (12, 25), (3, 14)]   # ← type the room's birthdays here

count = {}
for bday in our_birthdays:
    count[bday] = count.get(bday, 0) + 1

for bday, n in count.items():
    if n > 1:
        print('Match:', bday, 'is shared by', n, 'people')

---

## 1. Why Should I Care? + Review

### Why tuples?

A tuple is a **record that can't change**. Some data *should* never change while the program runs: a GPS location, an RGB colour, a date, the days in each month (from the warm-up). Putting them in a tuple gives you three things:

| | why it matters |
|---|---|
| **safety** | nobody (not even you, by accident) can change it, and Python stops you with a `TypeError` |
| **can be a dict key** | `(month, day)` and `(lat, lon)` can be dictionary keys, but a list can't |
| **lighter** | a tuple uses a bit less memory than the same list |

In [ ]:
import sys

inha = (37.45, 126.65)                  # GPS of Inha University: it should never change

places = {(37.45, 126.65): 'Inha University',
          (37.57, 126.98): 'Seoul City Hall'}
print(places[(37.45, 126.65)])          # a tuple works as a KEY

try:
    bad = {[37.45, 126.65]: 'Inha University'}
except TypeError as error:
    print('list as a key ->', error)    # a list can't be a key

try:
    inha[0] = 0
except TypeError as error:
    print('change a tuple ->', error)

print(sys.getsizeof([1, 2, 3, 4, 5]), 'bytes as a list')
print(sys.getsizeof((1, 2, 3, 4, 5)), 'bytes as a tuple')

### 1.1 Making a tuple

Round brackets and commas. A tuple can hold **any types**, even lists and other tuples.

**Trap:** `(20)` is just the number 20 in brackets. A tuple with **one** item needs a comma: `(20,)`. In fact the **comma** makes the tuple, so `seq = 1, 2, 3` is a tuple too.

In [ ]:
tuple1 = (1, 2, 3, 4, 5)
tuple2 = ('Math', 87, 'Arabic', 89.6)
tuple4 = (1, 2, 3, 4, 5, (10, 20))
print(tuple2)
print(tuple4)

tuple5 = (20)
print(type(tuple5))          # NOT a tuple!
tuple5 = (20,)
print(type(tuple5), tuple5)

seq = 1, 2, 3                # no brackets, still a tuple
print(type(seq), seq)

print(tuple([1, 2, 3]), tuple('abcde'))

### 1.2 Using a tuple

Everything that **reads** works like a list: indexing, slicing, `len`, `in`, `+`, `*`, `count`, `index`, `sorted`, `min`, `max`, `sum`.
Nothing **changes** a tuple. `+` makes a **new** tuple, and `sorted()` gives back a **list**.

The one surprise: a list **inside** a tuple can still change, because the tuple only holds on to the list, it doesn't freeze it.

In [ ]:
tuple1 = (2, 4, 6, 8, 10, 12)
print(len(tuple1), tuple1[0], tuple1[3], tuple1[-1], tuple1[1:4])

t6 = (1, 2, 3, 4, 5)
t6 = t6 + (6,)               # a NEW tuple, saved under the old name
print(t6)
print(('Hello',) * 3, 'Green' in ('Red', 'Green', 'Blue'))

tuple1 = (10, 20, 30, 10, 40, 10, 50)
print(tuple1.count(10), tuple1.index(30))

tuple1 = (19, 12, 56, 18, 9, 87, 34)
print(sorted(tuple1), min(tuple1), max(tuple1), sum(tuple1))

tup2 = (1, 2, 3, [8, 9])
tup2[3][1] = 10              # changes the LIST inside, which is allowed
print(tup2)

### 1.3 Unpacking — the most useful tuple trick

Give each item its own name in one line. Functions use this to **return several values**, and you'll use it in every dictionary loop (`for name, score in ...`).

In [ ]:
inha = (37.45, 126.65)
lat, lon = inha
print('latitude:', lat, ' longitude:', lon)


def hours_minutes(total_minutes):
    return total_minutes // 60, total_minutes % 60     # returns a tuple


h, m = hours_minutes(135)
print('The lecture is', h, 'h', m, 'min')

a, b = 1, 2
a, b = b, a                  # swap in one line
print(a, b)

### Why dictionaries? — The phone-book race

Imagine your university keeps **1,000,000 students**. You type a student ID and want the record **now**.

- A **list** has to check the items **one by one** until it finds the ID.
- A **dictionary** jumps **straight to** the key, like opening a phone book at the right letter.

Let's race them. The chart shows how long one search takes as the number of students grows.

In [ ]:
import time
import matplotlib.pyplot as plt

sizes = [100_000, 200_000, 300_000, 400_000, 500_000, 600_000, 700_000, 800_000, 900_000, 1_000_000]
list_ms = []
dict_ms = []

for n in sizes:
    ids = list(range(n))                 # a list of n student IDs
    book = {}
    for student_id in ids:               # a dict with the same IDs as keys
        book[student_id] = 'student'
    target = n - 1                       # look for the LAST student

    start = time.perf_counter()
    for repeat in range(20):
        target in ids
    list_ms.append((time.perf_counter() - start) / 20 * 1000)

    start = time.perf_counter()
    for repeat in range(20):
        target in book
    dict_ms.append((time.perf_counter() - start) / 20 * 1000)

print('1,000,000 students: list', round(list_ms[-1], 3), 'ms, dict', round(dict_ms[-1], 5), 'ms')
print('The dict was about', round(list_ms[-1] / dict_ms[-1]), 'times faster')

plt.plot(sizes, list_ms, marker='o', label='list: checks one by one')
plt.plot(sizes, dict_ms, marker='o', label='dict: jumps to the key')
plt.ticklabel_format(style='plain', axis='x')
plt.xlabel('number of students')
plt.ylabel('time for one search (ms)')
plt.title('Find one student: list vs dictionary')
plt.legend()
plt.show()

That's why **logins, contact lists, shop inventories, game save files and word counters** all use dictionaries. Whenever you **look something up by a name or an ID**, think *dict*.

### 1.4 Making a dictionary

A dictionary maps **keys** to **values**: `{key: value, key: value}`.
- Keys must be **unique** and **unchangeable** (strings, numbers, tuples, but **not** lists).
- Values can be **anything**, and they can repeat.

In [ ]:
dict1 = {'Mohsen': 95, 'Rajab': 89, 'badr': 92, 'Ayman': 85}
print(dict1)
print(dict1['Rajab'])

dic = {'NUM': 45, 1000: 'coding', 2000: 4.5, 3.4: 7, 'XXX': 89}   # keys of different types
print(dic[1000], dic[3.4])

### 1.5 Looking things up — `d[key]` vs `d.get(key)`

| code | key is there | key is missing |
|---|---|---|
| `d[key]` | the value | `KeyError` |
| `d.get(key)` | the value | `None` |
| `d.get(key, default)` | the value | `default` |

In [ ]:
dict1 = {'Mohsen': 95, 'Rajab': 89, 'badr': 92, 'Ayman': 85}
print(dict1.get('Ayman'))
print(dict1.get('Shyama'))
print(dict1.get('Shyama', 0))

try:
    print(dict1['Shyama'])
except KeyError as error:
    print('KeyError:', error)

### 1.6 Adding, changing, deleting — and `in`

- `d[key] = value` **adds** the key if it's new, and **overwrites** the value if the key already exists.
- `del d[key]` deletes it. `d.pop(key)` deletes it **and gives you the value**.
- `key in d` checks the **keys only**, never the values.
- Dictionaries **can't** be joined with `+` or repeated with `*`.

In [ ]:
dic = {'first': 1, 'second': 2, 'third': 3}
dic['fourth'] = 4            # new key → added
dic['second'] = 8            # existing key → overwritten
print(dic)

del dic['first']
x = dic.pop('second')
print(x, dic)

print('third' in dic, 3 in dic, 'Fourth' not in dic)

### 1.7 `keys()`, `values()`, `items()` — and looping

`items()` gives `(key, value)` **tuples**, so unpacking from 1.3 lets you name both at once: `for key, value in d.items():`.

In [ ]:
ODD = {1: 'One', 3: 'Three', 5: 'Five', 7: 'Seven', 9: 'Nine'}
print(ODD.keys())
print(ODD.values())
print(ODD.items())
print(len(ODD), ODD.get(9))

del ODD[9]
for number, word in ODD.items():
    print(number, '->', word)

ODD.clear()
print(ODD)

### Cheat sheet

| | **tuple** `(1, 2)` | **list** `[1, 2]` | **dict** `{'a': 1}` |
|---|---|---|---|
| change it? | no, never | yes | yes |
| find an item by | position `t[0]` | position `l[0]` | **key** `d['a']` |
| one item | `(20,)` | `[20]` | `{'a': 1}` |
| use it for | fixed records, dict keys, returning many values | a collection that grows and shrinks | look-ups, counting, totals |

---

## 2. Tuple Practice

### Predict T1 — Tuple or not?

Write your prediction on paper, **then** run it.

In [ ]:
a = (20)
b = (20,)
print(type(a), type(b))

t = (5, 3, 9)
s = sorted(t)
print(s, type(s))

t2 = t + (1,)
print(t, t2)
print(len(((1, 2), (3, 4))))

### Debug T1 — `make_redder`

A drawing app keeps colours as RGB tuples `(red, green, blue)`. This function should return the same colour with red turned up to the maximum, 255.

**Expected:** `make_redder((30, 144, 255))` ➞ `(255, 144, 255)`

In [ ]:
def make_redder(rgb):
    rgb[0] = 255
    return rgb

print(make_redder((30, 144, 255)))

### Write T1 — `score_summary`

Write `score_summary(scores)`, which returns a **tuple** `(lowest, highest, average)`, with the average rounded to 1 decimal place. Then **unpack** it into three names.

- `score_summary((70, 95, 60))` ➞ `(60, 95, 75.0)`
- `score_summary([88])` ➞ `(88, 88, 88.0)`
- `low, high, avg = score_summary((70, 95, 60, 81))` → `print(low, high, avg)` ➞ `60 95 76.5`

*Hint: `min`, `max`, `sum`, `len` and `round(x, 1)`. A `return` with commas returns a tuple.*

In [ ]:
# YOUR CODE HERE — define score_summary(scores)


print(score_summary((70, 95, 60)))
print(score_summary([88]))
low, high, avg = score_summary((70, 95, 60, 81))
print(low, high, avg)

---

## 3. Dictionary Practice

### Predict D1 — The campus café menu

In [ ]:
prices = {'kimbap': 3500, 'ramyeon': 4000, 'kimbap': 3000}
print(prices)
print(len(prices))
print('ramyeon' in prices, 4000 in prices)
print(prices.get('tteokbokki'), prices.get('tteokbokki', 'sold out'))

### Predict D2 — What does a loop over a dict give you?

In [ ]:
dict1 = {'Mohsen': 95, 'Rajab': 89, 'Ayman': 85}

for x in dict1:
    print(x)

for x in dict1.values():
    print(x, end=' ')
print()

for name, score in dict1.items():
    if score > 88:
        print(name, 'passed with', score)

### Debug D1 — `count_votes`

The class is voting on the pizza for the final-project party.

**Expected:** `{'pepperoni': 3, 'cheese': 1, 'bulgogi': 1}`

In [ ]:
def count_votes(votes):
    counts = {}
    for v in votes:
        counts[v] += 1
    return counts

print(count_votes(['pepperoni', 'cheese', 'pepperoni', 'bulgogi', 'pepperoni']))

### Debug D2 — Copied from the slides

This dictionary was copied **exactly** from the lecture slides. Why can't Python find `'badr'`?

**Expected:** `92`

In [ ]:
dict1 = {'Mohsen': 95, 'Rajab': 89, 'badr ': 92, 'Ayman': 85}
print(dict1['badr'])

### Debug D3 — Print the grade report

**Expected:**
```
Mohsen 95
Rajab 89
badr 92
Ayman 85
```

In [ ]:
grades = {'Mohsen': 95, 'Rajab': 89, 'badr': 92, 'Ayman': 85}
for name, score in grades:
    print(name, score)

### Write D1 — `order_total`

The campus café keeps its menu as a dict `{item: price in won}`. Write `order_total(menu, order)` that returns the total price of a list of items. Items that are **not on the menu** are skipped.

- `order_total(menu, ['kimbap', 'coffee', 'coffee'])` ➞ `7500`
- `order_total(menu, ['ramyeon', 'pizza'])` ➞ `4000` *(no pizza here)*
- `order_total(menu, [])` ➞ `0`

In [ ]:
menu = {'kimbap': 3500, 'ramyeon': 4000, 'tteokbokki': 4500, 'coffee': 2000}

# YOUR CODE HERE — define order_total(menu, order)


print(order_total(menu, ['kimbap', 'coffee', 'coffee']))
print(order_total(menu, ['ramyeon', 'pizza']))
print(order_total(menu, []))

### Write D2 — `top_student`

Write `top_student(scores)`, which returns the best student as a **tuple** `(name, score)`. Return `None` if the dict is empty.

- `top_student({'Mohsen': 95, 'Rajab': 89, 'badr': 92})` ➞ `('Mohsen', 95)`
- `top_student({'Ayman': 85})` ➞ `('Ayman', 85)`
- `top_student({})` ➞ `None`

*Hint: loop over `.items()` and remember the best pair so far.*

In [ ]:
# YOUR CODE HERE — define top_student(scores)


print(top_student({'Mohsen': 95, 'Rajab': 89, 'badr': 92}))
print(top_student({'Ayman': 85}))
print(top_student({}))

### Write D3 — `spending_by_category`

Your banking app gives you a list of payments as **tuples** `(category, amount)`. Write `spending_by_category(expenses)`, which returns a **dict** with the **total** for each category.

```python
expenses = [('food', 12000), ('transport', 1450), ('food', 8500), ('books', 23000),
            ('transport', 1450), ('food', 9000), ('coffee', 4500), ('coffee', 4800)]
```
➞ `{'food': 29500, 'transport': 2900, 'books': 23000, 'coffee': 9300}`

`spending_by_category([])` ➞ `{}`

*Hint: it's Debug D1's counting pattern, but you add the **amount** instead of 1. We'll chart this result in Part 4.*

In [ ]:
expenses = [('food', 12000), ('transport', 1450), ('food', 8500), ('books', 23000),
            ('transport', 1450), ('food', 9000), ('coffee', 4500), ('coffee', 4800)]

# YOUR CODE HERE — define spending_by_category(expenses)


print(spending_by_category(expenses))
print(spending_by_category([]))

---

## 4. Matplotlib — Why and How

### Why charts?

Quick: in the dict below, **which month is the warmest**, and **when does it start getting cold**? Try to answer from the printed numbers first, then look at the chart.

In [ ]:
import matplotlib.pyplot as plt

seoul_temp = {'Jan': -2.0, 'Feb': 0.6, 'Mar': 6.1, 'Apr': 12.6, 'May': 18.2, 'Jun': 22.7,
              'Jul': 25.3, 'Aug': 26.1, 'Sep': 21.6, 'Oct': 14.8, 'Nov': 7.2, 'Dec': 0.4}   # average °C (approx.)
print(seoul_temp)

plt.plot(seoul_temp.keys(), seoul_temp.values(), marker='o')
plt.xlabel('month')
plt.ylabel('average temperature (°C)')
plt.title('Seoul: average temperature by month')
plt.show()

Your eyes find the pattern in a chart in **one second**. In a table of numbers they have to read every value. That is why every app, report and news article shows charts, and why the warm-up's curve convinced you faster than any list of 60 numbers would have.

| question | chart | real example |
|---|---|---|
| how does it change over **time**? | **line** `plt.plot` | stock prices, your typing speed, temperature |
| which **category** is biggest? | **bar** `plt.bar` | sales per menu item, spending per category |
| what **share** of the whole? | **pie** `plt.pie` | where your budget goes |
| are two things **related**? | **scatter** `plt.scatter` | study hours vs exam score |
| how are values **spread out**? | **histogram** `plt.hist` | the exam score distribution of a class |

**Final project:** matplotlib is on the list of allowed libraries. A chart of your saved scores or data makes a demo look *much* stronger.

### The recipe (every chart)

```python
import matplotlib.pyplot as plt   # 1. import (once)
plt.plot(x_values, y_values)      # 2. draw: plot / bar / pie / scatter / hist
plt.xlabel('...')                 # 3. label it: x, y, title (legend if 2+ lines)
plt.ylabel('...')
plt.title('...')
plt.show()                        # 4. show it, once per chart
```

### 4.1 Line chart — my TypeSprint progress

You practise typing for 10 days and save your words-per-minute each day. Is it working?

In [ ]:
import matplotlib.pyplot as plt

days = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
wpm = [28, 31, 30, 34, 37, 36, 40, 43, 42, 46]

plt.plot(days, wpm)
plt.xlabel('practice day')
plt.ylabel('words per minute')
plt.title('My TypeSprint progress')
plt.show()

### 4.2 Two lines + style — who improves faster?

Give each line a `label=...`, then `plt.legend()` shows which line is which. You can also style a line with `color`, `linestyle`, `linewidth`, `marker` and `markersize`.

In [ ]:
days = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mina = [28, 31, 30, 34, 37, 36, 40, 43, 42, 46]
jun = [35, 36, 38, 37, 39, 40, 40, 41, 43, 42]

plt.plot(days, mina, label='Mina', color='green', linestyle='dashed', linewidth=3,
         marker='o', markerfacecolor='blue', markersize=8)
plt.plot(days, jun, label='Jun', color='orange', marker='s')
plt.ylim(20, 50)
plt.xlabel('practice day')
plt.ylabel('words per minute')
plt.title('Mina vs Jun')
plt.legend()
plt.show()

### 4.3 Bar chart — where did my money go? (+ `savefig`)

A bar chart is made for a **dictionary**: the keys go on the x-axis and the values are the bar heights. This is Write D3's result. `plt.text(x, y, text)` writes each amount on top of its bar, and `plt.savefig()` saves the chart as an image you can put in a report. Call it **before** `plt.show()`.

In [ ]:
budget = {'food': 29500, 'transport': 2900, 'books': 23000, 'coffee': 9300}   # from Write D3

plt.bar(budget.keys(), budget.values(), color='green')
for category, amount in budget.items():
    plt.text(category, amount, amount, ha='center', va='bottom')
plt.xlabel('category')
plt.ylabel('won')
plt.title('My spending this week')
plt.savefig('budget.png')
plt.show()

### 4.4 Pie chart — what share of my money?

Same dict, different question: not "how much", but "**what part of the total**". `autopct='%1.1f%%'` writes the percentages, and `explode` (a tuple!) pulls one slice out.

In [ ]:
plt.pie(budget.values(), labels=budget.keys(), autopct='%1.1f%%',
        startangle=90, explode=(0.1, 0, 0, 0))
plt.title('My spending this week')
plt.show()

### 4.5 Scatter plot — does studying help?

Each student is a **tuple** `(hours studied, exam score)`. One dot per student. If the dots go up from left to right, the two are related.

In [ ]:
students = [(1, 52), (2, 58), (2, 65), (3, 61), (4, 70), (4, 74), (5, 69), (5, 80),
            (6, 78), (7, 85), (7, 79), (8, 88), (9, 91), (10, 87), (10, 95)]

hours = []
scores = []
for h, s in students:            # unpack each tuple
    hours.append(h)
    scores.append(s)

plt.scatter(hours, scores, color='green', marker='*', s=80)
plt.xlabel('hours studied')
plt.ylabel('exam score')
plt.title('Does studying help?')
plt.show()

### 4.6 Histogram — how did the class do?

A histogram puts values into **bins** (0–10, 10–20, …) and counts how many fall into each one.

**Trap from the slides:** `range = (0, 100)` on its own line **replaces** Python's `range()`, and your next `for i in range(...)` crashes. Pass it **inside** the call, as below.

In [ ]:
exam_scores = [45, 52, 58, 61, 63, 65, 67, 68, 70, 71, 72, 72, 74, 75, 76, 77, 78, 78, 79, 80,
               81, 82, 83, 84, 85, 85, 86, 88, 89, 90, 91, 92, 94, 95, 97, 38, 55, 66, 73, 87]

plt.hist(exam_scores, bins=10, range=(0, 100), color='green', rwidth=0.8)
plt.xlabel('exam score')
plt.ylabel('number of students')
plt.title('Midterm scores')
plt.show()

---

## 5. Chart Challenge

> *Three clients need charts by the end of class. A strict checker reviews every chart: right **type**, right **data**, and a **title**, **axis labels** and a **legend** when needed. Fastest team wins!*

**Rules**
1. Sit with your **final-project team**. One laptop types and everyone else helps.
2. Run the **engine cell**, then `start('your team name')`. The clock starts.
3. In each mission cell, draw the chart, then call `check('M1')` (etc.) **instead of** `plt.show()`. It shows your chart and prints PASS, or tells you what's missing.
4. Finish M1, M2 and M3 to get the finish banner with your time. Show it to the TA. Done early? Try the **BONUS**.

In [ ]:
# CHART CHALLENGE ENGINE: run this cell once and don't change it.
import time
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Wedge

_M1 = {'bibimbap': 42, 'kimbap': 35, 'ramyeon': 58, 'tonkatsu': 27, 'salad': 12}
_M2 = ([2, 5, 9, 14, 20], [4, 6, 7, 12, 22])
_M3 = {'food': 47000, 'printing': 12000, 'software': 12000, 'transport': 9000}
_M3_PAYMENTS = 7          # slices you get if you forget to total the categories
_state = {'team': None, 'start': None, 'done': set(), 'banner': False}


def start(team):
    _state.update(team=team, start=time.time(), done=set(), banner=False)
    print('Team', team, '- the clock is running. Missions: M1, M2, M3 (+ BONUS)')


def _labels(ax, pie=False):
    missing = []
    if not ax.get_title():
        missing.append('a title: plt.title(...)')
    if not pie and not ax.get_xlabel():
        missing.append('an x-axis label: plt.xlabel(...)')
    if not pie and not ax.get_ylabel():
        missing.append('a y-axis label: plt.ylabel(...)')
    return missing


def _check_m1(ax):
    bars = [p for p in ax.patches if isinstance(p, Rectangle)]
    if not bars:
        return ['bars! This mission needs plt.bar(...)']
    heights = sorted(round(b.get_height()) for b in bars)
    if heights != sorted(_M1.values()):
        return ['the right bar heights. Use lunch_sales.keys() and lunch_sales.values()']
    return _labels(ax)


def _check_m2(ax):
    lines = [[round(float(y)) for y in line.get_ydata()] for line in ax.get_lines()]
    if len(lines) < 2:
        return ['TWO lines: call plt.plot(...) once for each team']
    if _M2[0] not in lines or _M2[1] not in lines:
        return ['lines with the right data: weeks on x, team_a and team_b on y']
    missing = _labels(ax)
    legend = ax.get_legend()
    if legend is None or len(legend.get_texts()) < 2:
        missing.append('a legend: give each plt.plot(...) a label=..., then plt.legend()')
    return missing


def _check_m3(ax):
    wedges = [p for p in ax.patches if isinstance(p, Wedge)]
    if not wedges:
        return ['a pie! This mission needs plt.pie(...)']
    if len(wedges) == _M3_PAYMENTS:
        return ['ONE slice per category. Some categories appear twice: add them up into a dict first']
    if len(wedges) != len(_M3):
        return ['one slice per category (' + str(len(_M3)) + ' categories)']
    total = sum(_M3.values())
    shares = sorted(round((w.theta2 - w.theta1) / 360, 2) for w in wedges)
    if shares != sorted(round(v / total, 2) for v in _M3.values()):
        return ['the right slice sizes. Use the TOTAL of each category']
    return _labels(ax, pie=True)


def _check_bonus(fig):
    if len(fig.axes) != 2:
        return ['2 charts in one figure: plt.subplot(2, 1, 1) and plt.subplot(2, 1, 2)']
    top, bottom = fig.axes
    return _check_m1(top) + _check_m2(bottom)


def check(mission):
    if _state['start'] is None:
        print('Run start(...) first!')
        return
    fig = plt.gcf()
    if not fig.axes:
        plt.close(fig)
        print('NOT YET: I see no chart. Draw it first, and call check(...) INSTEAD of plt.show().')
        return
    mission = mission.upper()
    if mission == 'M1':
        missing = _check_m1(plt.gca())
    elif mission == 'M2':
        missing = _check_m2(plt.gca())
    elif mission == 'M3':
        missing = _check_m3(plt.gca())
    elif mission == 'BONUS':
        missing = _check_bonus(fig)
    else:
        print("Unknown mission. Use 'M1', 'M2', 'M3' or 'BONUS'.")
        return
    plt.show()
    plt.close(fig)
    if missing:
        print('NOT YET:', mission, '- your chart still needs:')
        for item in missing:
            print('   -', item)
        return
    print('PASS:', mission, 'approved')
    _state['done'].add(mission)
    if {'M1', 'M2', 'M3'} <= _state['done'] and not _state['banner']:
        _state['banner'] = True
        seconds = int(time.time() - _state['start'])
        print()
        print('=' * 44)
        print('   ALL CHARTS APPROVED — Team', _state['team'])
        print('   time:', seconds // 60, 'min', seconds % 60, 's')
        print('=' * 44)

In [ ]:
start('your team name')   # ← put your team's name here

### Mission 1 — The cafeteria manager

*"Which lunch sells best? I need to know what to cook less of."*

Draw a **bar chart** of `lunch_sales` (dish → plates sold today), with a title and both axis labels.

In [ ]:
lunch_sales = {'bibimbap': 42, 'kimbap': 35, 'ramyeon': 58, 'tonkatsu': 27, 'salad': 12}

# YOUR CODE HERE: draw the bar chart, title and labels

check('M1')

### Mission 2 — The professor

*"Two project teams report how many features they finished by the end of each week. Who's ahead?"*

Draw **two lines** on one chart: weeks on the x-axis, one line per team, with a **legend**, a title and axis labels.

In [ ]:
weeks = [1, 2, 3, 4, 5]
team_a = [2, 5, 9, 14, 20]
team_b = [4, 6, 7, 12, 22]

# YOUR CODE HERE: two lines, labels, title, legend

check('M2')

### Mission 3 — Your own team

*"Where did our final-project budget go?"*

The payments are a list of `(category, amount)` **tuples**, and some categories appear **more than once**. Total them into a **dict** first (Write D3 helps!), then draw a **pie chart** with labels, percentages and a title.

In [ ]:
team_expenses = [('food', 25000), ('printing', 8000), ('software', 12000), ('food', 22000),
                 ('transport', 4500), ('printing', 4000), ('transport', 4500)]

# YOUR CODE HERE: total per category in a dict, then the pie chart and a title

check('M3')

### BONUS — One dashboard

Put Mission 1 (bar chart) **on top** and Mission 2 (two lines) **below**, in one figure: `plt.figure(figsize=(8, 8))`, then `plt.subplot(2, 1, 1)` before the first chart and `plt.subplot(2, 1, 2)` before the second. Every chart still needs its labels.

In [ ]:
# YOUR CODE HERE: the BONUS dashboard

check('BONUS')

---

## 6. Debrief — The Four Traps of the Week

| trap | what happens | do this instead |
|---|---|---|
| `t = (20)` | it's an **int**, not a tuple | `t = (20,)`: the comma makes the tuple |
| `d[key]` on a missing key | `KeyError` (counting!) | `d.get(key, 0)`, or check `key in d` first |
| `for a, b in d:` | a loop over a dict gives **keys only** | `for key, value in d.items():` |
| a chart with no title/labels | nobody knows what it shows | `title`, `xlabel`, `ylabel`, and `legend` for 2+ lines |

**Final project:** keep your data in a dict (`{player: high score}`, `{task: done?}`, `{word: count}`), save it with last week's file functions, and show it with one chart in your demo.

---

## Take-Home (optional, not done in class) — Problems #1 and #2 from the slides

The slides make random data with `pandas` and `numpy`. Here we use plain lists and `random`, which gives the same charts.

In [ ]:
import random
import matplotlib.pyplot as plt

x = list(range(30))
a = [random.randint(0, 100) for i in x]
b = [random.randint(0, 100) for i in x]
print(a)
print(b)

### Problem #1

1. Create a figure of size 15×8 with two subplots, top and bottom.
2. Draw two lines in the top axes, one green and one orange.
3. Add a legend for the plot (Green and Orange), and put it in the top-middle of the graph. *(Hint: `plt.legend(loc='upper center')`.)*
4. In the bottom axes, draw only the data points, marked by circles, with no line connecting them. *(Hint: `plt.plot(x, a, 'o')`.)*

In [ ]:
# YOUR CODE HERE — Problem #1

### Problem #2

1. Create a figure with three subplots, one above the other.
2. Draw a bar graph of `a` in the first and of `b` in the second. In the last one, draw both `a` and `b`, with `b` **on top of** `a`. *(Hint: `plt.bar(x, b, bottom=a)`.)*
3. Write the size of each bar on top of it. *(Hint: `plt.text(i, height, height, ha='center')` in a loop.)*

In [ ]:
# YOUR CODE HERE — Problem #2

### Plotting curves with `numpy`

`numpy` (already in Colab) makes many x-values at once and applies a math function to all of them.

In [ ]:
import numpy as np

x = np.arange(0, 2 * np.pi, 0.1)
y = np.sin(x)

plt.plot(x, y)
plt.title('sin(x)')
plt.show()

---

## What We Practiced

- **Why:** tuples are safe fixed records that can be dict keys, dictionaries find things **instantly** by key, and charts show a pattern in **one second**
- Tuples: `(20,)` vs `(20)`, indexing and slicing, `count`, `index`, `sorted` (gives a list), `+` makes a **new** tuple, and **unpacking** `a, b = t`
- Dictionaries: `d[key]` vs `d.get(key, default)`, adding and overwriting, `del`, `pop`, `in` (keys only), `keys()`, `values()`, `items()`
- The **counting/total pattern**: `d[k] = d.get(k, 0) + amount`
- Matplotlib: `plot`, `bar`, `pie`, `scatter`, `hist`, plus `xlabel`, `ylabel`, `title`, `legend`, `savefig`, `subplot` and `show`